# **Assignment 08: Stable WAN 2.1**

**Available:** Oct 16, 2025 3:00pm until Oct 25, 2025 11:59pm

**Details**
- Dataset: https://www.kaggle.com/datasets/sharjeelmazhar/human-activity-recognition-video-dataset
- Train LoRA for WAN 2.1 1.3G model (not the 14G model, too big)​
- Generate videos for human activity recognition, 10 per category.​
- Grader will judge the quality and gives grade


## **Setup**

In [13]:
## Import Libraries

# Set CUDA_VISIBLE_DEVICES to make both GPUs visible
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

# Install all packages for from the requirements.txt
%pip install -r requirements.txt

import torch
import torch.nn as nn
import torchvision
import datasets
import cv2
import matplotlib.pyplot as plt
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch import optim
from tqdm.notebook import tqdm
from torchinfo import summary
import einops
import PIL
import numpy as np
import pandas as pd
# Use a pipeline as a high-level helper
from transformers import pipeline
import time
import psutil
import gc
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import json
from collections import defaultdict
import numpy as np

# Authorize Huggingface account
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv('/mnt/Storage02/SoftwareDev/CAP_6411_Assignments/.env')

# Get Hugging Face token
hf_token = os.getenv('HUGGINGFACE_HUB_TOKEN') or os.getenv('HF_TOKEN')

if hf_token:
    print("Found Hugging Face token in environment variables")
    
    
    from huggingface_hub import login, whoami
    
    try:
        # Login to Hugging Face Hub
        login(token=hf_token)
        
        # Verify login by getting user info
        user_info = whoami()
        print(f"Successfully authenticated with Hugging Face!")
        print(f"Logged in as: {user_info['name']}")
        
        # Set the token as environment variable for other libraries
        os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
        os.environ['HF_TOKEN'] = hf_token
        
    except Exception as e:
        print(f"Authentication failed: {e}")
        print("Will proceed without pre-trained models if needed")
        hf_token = None
else:
    print("No Hugging Face token found in .env file")
    print("Please add HUGGINGFACE_HUB_TOKEN=your_token_here to your .env file")
    hf_token = None


# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")


Note: you may need to restart the kernel to use updated packages.
Found Hugging Face token in environment variables
Note: you may need to restart the kernel to use updated packages.
Found Hugging Face token in environment variables


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Successfully authenticated with Hugging Face!
Logged in as: malneyugnfl


In [14]:
# GPU Setup 
# Comprehensive GPU diagnostics
print("\n=== GPU Diagnostics ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs detected: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print("\n=== All Available GPUs ===")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}:")
        print(f"  Name: {props.name}")
        print(f"  Total Memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"  Multi-processor count: {props.multi_processor_count}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print()

# Device selection with preference for cuda:1 (A6000) -> cuda:0 (4090) -> mps (Apple Silicon) -> cpu
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    device = torch.device('cuda:1')  # This should now be your A6000!
    print(f"Using GPU 1: {torch.cuda.get_device_name(1)}")
elif torch.cuda.is_available():
    device = torch.device('cuda:0')
    print(f"Using GPU 0: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using Apple Silicon MPS")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"Selected device: {device}")

# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")

def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024

def get_gpu_memory_usage():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024 / 1024
    elif device.type == 'mps':
        # MPS doesn't have direct memory monitoring like CUDA
        # Return 0 as a placeholder
        return 0
    return 0


=== GPU Diagnostics ===
PyTorch version: 2.8.0+cu128
CUDA available: True
MPS available: False
CUDA version: 12.8
Number of GPUs detected: 2

=== All Available GPUs ===
GPU 0:
  Name: NVIDIA GeForce RTX 4090 Laptop GPU
  Total Memory: 15.70 GB
  Multi-processor count: 76
  Compute Capability: 8.9

GPU 1:
  Name: NVIDIA RTX A6000
  Total Memory: 47.53 GB
  Multi-processor count: 84
  Compute Capability: 8.6

Using GPU 1: NVIDIA RTX A6000
Selected device: cuda:1


In [15]:
# Function to clear memory for both CUDA and MPS
def clear_memory():
    """Clear CPU and GPU memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()
    print(f"Cleared memory. Current CPU memory usage: {get_memory_usage():.2f} MB, GPU memory usage: {get_gpu_memory_usage():.2f} MB")

## **Import Model**

In [16]:
## Source: https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B

import os
import shutil

# Clone the WAN 2.1 repository only if it doesn't already exist
if not os.path.exists('Wan2.1'):
    print("Cloning WAN 2.1 repository...")
    !git clone https://github.com/Wan-Video/Wan2.1.git
    
    # Deactivate the cloned repository by removing .git folder
    git_folder = os.path.join('Wan2.1', '.git')
    if os.path.exists(git_folder):
        print("Deactivating cloned repository (removing .git folder)...")
        shutil.rmtree(git_folder)
        print("✅ Wan2.1 is no longer a git repository")
    else:
        print("No .git folder found in cloned repository")
else:
    print("WAN 2.1 repository already exists, skipping clone.")

# Change to the Wan2.1 directory
os.chdir('Wan2.1')
print(f"Changed to directory: {os.getcwd()}")

# Install requirements (ensure torch >= 2.4.0)
!pip install -r requirements.txt

# Download the model from Hugging Face only if it doesn't already exist
if not os.path.exists('./Wan2.1-T2V-1.3B'):
    print("Downloading WAN 2.1 model...")
    !huggingface-cli download Wan-AI/Wan2.1-T2V-1.3B --local-dir ./Wan2.1-T2V-1.3B
else:
    print("WAN 2.1 model already exists, skipping download.")

WAN 2.1 repository already exists, skipping clone.
Changed to directory: /mnt/Storage02/SoftwareDev/CAP_6411_Assignments/Aug2025/01_Assignments/08_WAN_2_1/Wan2.1
WAN 2.1 model already exists, skipping download.
WAN 2.1 model already exists, skipping download.


## **Implement LORA into WAN 2.1**

In [17]:
# Implement LORA into WAN 2.1 model:

import sys
sys.path.append('./Wan2.1')

from wan.modules.model import WanModel
from peft import LoraConfig, get_peft_model, TaskType
import torch.nn as nn
from typing import List, Dict, Any

def create_lora_wan_model(model_config: Dict[str, Any], lora_config: Dict[str, Any] = None):
    """
    Create a WAN 2.1 model with LoRA adapters
    
    Args:
        model_config: Configuration dictionary for the WAN model
        lora_config: Configuration dictionary for LoRA adapters
    
    Returns:
        WAN model with LoRA adapters attached
    """
    
    # Default LoRA configuration if not provided
    if lora_config is None:
        lora_config = {
            "r": 16,  # rank
            "lora_alpha": 32,  # scaling factor
            "lora_dropout": 0.1,  # dropout probability
            "target_modules": [
                # Self-attention layers
                "q", "k", "v", "o",  # in WanSelfAttention
                # Cross-attention layers  
                "k_img", "v_img",   # in WanI2VCrossAttention (if applicable)
                # FFN layers
                "ffn.0", "ffn.2",   # first and third layers in FFN
                # Projection layers
                "head.head",        # output head
                "text_embedding.0", "text_embedding.2"  # text embedding layers
            ]
        }
    
    # Create the base WAN model
    model = WanModel(**model_config)
    
    # Configure LoRA
    peft_config = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,  # Using feature extraction as base task type
        inference_mode=False,
        r=lora_config["r"],
        lora_alpha=lora_config["lora_alpha"], 
        lora_dropout=lora_config["lora_dropout"],
        target_modules=lora_config["target_modules"],
        bias="none"  # Don't adapt bias parameters
    )
    
    # Apply LoRA to the model
    model = get_peft_model(model, peft_config)
    
    return model

def print_trainable_parameters(model):
    """
    Print the number of trainable parameters in the model
    """
    trainable_params = 0
    all_param = 0
    
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    
    print(f"Trainable params: {trainable_params:,} || "
          f"All params: {all_param:,} || "
          f"Trainable%: {100 * trainable_params / all_param:.2f}")

# Example usage - Create WAN 2.1 model with LoRA
print("Creating WAN 2.1 model with LoRA adapters...")

# Model configuration for WAN 2.1 1.3B (smaller version)
wan_config = {
    "model_type": "t2v",  # text-to-video
    "patch_size": (1, 2, 2),
    "text_len": 512,
    "in_dim": 16,
    "dim": 2048,        # Reduced from 2048 for 1.3B version
    "ffn_dim": 8192,    # Reduced accordingly  
    "freq_dim": 256,
    "text_dim": 4096,
    "out_dim": 16,
    "num_heads": 16,    # Reduced from 32
    "num_layers": 24,   # Reduced from 32 for 1.3B version
    "window_size": (-1, -1),
    "qk_norm": True,
    "cross_attn_norm": True,
    "eps": 1e-6
}

# LoRA configuration - you can adjust these hyperparameters
lora_config = {
    "r": 16,              # Rank - higher = more parameters but better adaptation
    "lora_alpha": 32,     # Scaling factor - typically 2x the rank
    "lora_dropout": 0.1,  # Dropout for regularization
    "target_modules": [
        # Self-attention components in each block
        "blocks.*.self_attn.q",
        "blocks.*.self_attn.k", 
        "blocks.*.self_attn.v",
        "blocks.*.self_attn.o",
        # Cross-attention components
        "blocks.*.cross_attn.q",
        "blocks.*.cross_attn.k",
        "blocks.*.cross_attn.v", 
        "blocks.*.cross_attn.o",
        # FFN components
        "blocks.*.ffn.0",  # First linear layer in FFN
        "blocks.*.ffn.2",  # Second linear layer in FFN
        # Head projection
        "head.head",
        # Text embedding layers
        "text_embedding.0",
        "text_embedding.2"
    ]
}

try:
    # Create the model with LoRA
    wan_lora_model = create_lora_wan_model(wan_config, lora_config)
    
    # Move to appropriate device
    wan_lora_model = wan_lora_model.to(device)
    
    print(f"✓ Successfully created WAN 2.1 model with LoRA adapters on {device}")
    
    # Print parameter information
    print("\n" + "="*50)
    print("MODEL PARAMETER SUMMARY")
    print("="*50)
    print_trainable_parameters(wan_lora_model)
    
    # Print model structure (first few layers)
    print(f"\nModel structure preview:")
    print(f"Model type: {wan_lora_model.config.model_type}")
    print(f"Number of layers: {wan_lora_model.config.num_layers}")
    print(f"Hidden dimension: {wan_lora_model.config.dim}")
    print(f"Number of attention heads: {wan_lora_model.config.num_heads}")
    
    # Show some LoRA adapter info
    print(f"\nLoRA Configuration:")
    print(f"Rank (r): {lora_config['r']}")
    print(f"Alpha: {lora_config['lora_alpha']}")
    print(f"Dropout: {lora_config['lora_dropout']}")
    
    print(f"\n✓ WAN 2.1 with LoRA is ready for training!")
    
except Exception as e:
    print(f"❌ Error creating WAN model with LoRA: {str(e)}")
    print("This might be due to model architecture changes or missing dependencies.")
    import traceback
    traceback.print_exc()

# Clear some memory
clear_memory()

Creating WAN 2.1 model with LoRA adapters...
✓ Successfully created WAN 2.1 model with LoRA adapters on cuda:1

MODEL PARAMETER SUMMARY
Trainable params: 197,632 || All params: 1,654,795,328 || Trainable%: 0.01

Model structure preview:
Model type: t2v
Number of layers: 24
Hidden dimension: 2048
Number of attention heads: 16

LoRA Configuration:
Rank (r): 16
Alpha: 32
Dropout: 0.1

✓ WAN 2.1 with LoRA is ready for training!
Cleared memory. Current CPU memory usage: 3855.04 MB, GPU memory usage: 0.00 MB
✓ Successfully created WAN 2.1 model with LoRA adapters on cuda:1

MODEL PARAMETER SUMMARY
Trainable params: 197,632 || All params: 1,654,795,328 || Trainable%: 0.01

Model structure preview:
Model type: t2v
Number of layers: 24
Hidden dimension: 2048
Number of attention heads: 16

LoRA Configuration:
Rank (r): 16
Alpha: 32
Dropout: 0.1

✓ WAN 2.1 with LoRA is ready for training!
Cleared memory. Current CPU memory usage: 3855.04 MB, GPU memory usage: 0.00 MB


## **Dataset Preparation**

In [18]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import json
from pathlib import Path
import random
from tqdm import tqdm
import matplotlib.pyplot as plt

class HARVideoDataset(Dataset):
    """
    Human Action Recognition Video Dataset for WAN 2.1 training
    Processes videos into frames suitable for text-to-video generation training
    """
    
    def __init__(self, data_dir, split='train', max_frames=16, frame_size=(224, 224), 
                 train_ratio=0.8, random_seed=42):
        """
        Initialize the HAR Video Dataset
        
        Args:
            data_dir: Path to HAR_Video_Dataset directory
            split: 'train', 'val', or 'test'
            max_frames: Maximum number of frames to extract per video
            frame_size: Target frame size (height, width)
            train_ratio: Ratio of data to use for training
            random_seed: Random seed for reproducibility
        """
        self.data_dir = Path(data_dir)
        self.split = split
        self.max_frames = max_frames
        self.frame_size = frame_size
        self.random_seed = random_seed
        
        # Define action classes
        self.classes = [
            "Clapping",
            "Meet and Split", 
            "Sitting",
            "Standing Still",
            "Walking",
            "Walking While Reading Book",
            "Walking While Using Phone"
        ]
        
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        # Load and split dataset
        self.video_paths, self.labels, self.prompts = self._load_dataset()
        
        print(f"Loaded {len(self.video_paths)} videos for {split} split")
        print(f"Classes: {self.classes}")
        print(f"Class distribution: {dict(zip(self.classes, [self.labels.count(i) for i in range(len(self.classes))]))}")
    
    def _load_dataset(self):
        """Load video paths and create train/val splits"""
        all_videos = []
        all_labels = []
        all_prompts = []
        
        # Collect all videos
        for class_name in self.classes:
            class_dir = self.data_dir / class_name
            if not class_dir.exists():
                print(f"Warning: Class directory {class_dir} not found")
                continue
                
            video_files = list(class_dir.glob("*.mp4"))
            class_label = self.class_to_idx[class_name]
            
            for video_file in video_files:
                all_videos.append(str(video_file))
                all_labels.append(class_label)
                # Create descriptive prompts for each action
                prompt = self._create_prompt(class_name)
                all_prompts.append(prompt)
        
        # Split dataset
        if len(all_videos) == 0:
            raise ValueError("No videos found in dataset")
        
        # Create train/val/test splits
        train_videos, temp_videos, train_labels, temp_labels, train_prompts, temp_prompts = train_test_split(
            all_videos, all_labels, all_prompts, 
            train_size=0.7, random_state=self.random_seed, stratify=all_labels
        )
        
        val_videos, test_videos, val_labels, test_labels, val_prompts, test_prompts = train_test_split(
            temp_videos, temp_labels, temp_prompts,
            train_size=0.5, random_state=self.random_seed, stratify=temp_labels
        )
        
        # Return appropriate split
        if self.split == 'train':
            return train_videos, train_labels, train_prompts
        elif self.split == 'val':
            return val_videos, val_labels, val_prompts
        else:  # test
            return test_videos, test_labels, test_prompts
    
    def _create_prompt(self, class_name):
        """Create descriptive text prompts for each action class"""
        prompt_templates = {
            "Clapping": [
                "A person clapping their hands together",
                "Person applauding by clapping hands",
                "Human clapping with both hands",
                "Someone clapping their hands rhythmically"
            ],
            "Meet and Split": [
                "People meeting and then separating", 
                "Group of people gathering then splitting apart",
                "Individuals coming together and then going separate ways",
                "People meeting briefly then walking away"
            ],
            "Sitting": [
                "A person sitting down",
                "Person sitting in a seated position",
                "Human sitting calmly",
                "Someone sitting still"
            ],
            "Standing Still": [
                "A person standing motionless",
                "Person standing still without moving",
                "Human standing in place",
                "Someone standing upright and stationary"
            ],
            "Walking": [
                "A person walking forward",
                "Person walking at normal pace",
                "Human walking naturally",
                "Someone taking steps while walking"
            ],
            "Walking While Reading Book": [
                "A person walking while reading a book",
                "Person reading and walking simultaneously",
                "Human walking and holding a book to read",
                "Someone walking while focused on reading"
            ],
            "Walking While Using Phone": [
                "A person walking while using their phone",
                "Person walking and looking at mobile device",
                "Human walking while texting on phone", 
                "Someone walking while using smartphone"
            ]
        }
        
        return random.choice(prompt_templates[class_name])
    
    def _extract_frames(self, video_path):
        """Extract frames from video file"""
        cap = cv2.VideoCapture(video_path)
        frames = []
        
        # Get video properties
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        
        if total_frames == 0:
            cap.release()
            return None
        
        # Calculate frame indices to extract
        if total_frames <= self.max_frames:
            # Use all frames if video is short
            frame_indices = list(range(total_frames))
        else:
            # Sample frames uniformly across the video
            frame_indices = np.linspace(0, total_frames-1, self.max_frames, dtype=int)
        
        # Extract frames
        for frame_idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            
            if ret:
                # Convert BGR to RGB
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                # Resize frame
                frame = cv2.resize(frame, self.frame_size)
                # Normalize to [0, 1]
                frame = frame.astype(np.float32) / 255.0
                frames.append(frame)
        
        cap.release()
        
        if len(frames) == 0:
            return None
        
        # Pad or truncate to max_frames
        while len(frames) < self.max_frames:
            frames.append(frames[-1])  # Repeat last frame
        frames = frames[:self.max_frames]
        
        # Convert to tensor (T, H, W, C) -> (C, T, H, W)
        frames_tensor = torch.tensor(np.array(frames))
        frames_tensor = frames_tensor.permute(3, 0, 1, 2)  # (C, T, H, W)
        
        return frames_tensor
    
    def __len__(self):
        return len(self.video_paths)
    
    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        prompt = self.prompts[idx]
        
        # Extract frames
        frames = self._extract_frames(video_path)
        
        if frames is None:
            # Return a random valid sample if current video fails
            return self.__getitem__(random.randint(0, len(self) - 1))
        
        return {
            'frames': frames,
            'label': label,
            'prompt': prompt,
            'class_name': self.classes[label],
            'video_path': video_path
        }

def create_har_dataloaders(data_dir, batch_size=4, max_frames=16, frame_size=(224, 224), num_workers=2):
    """
    Create train, validation, and test dataloaders for HAR dataset
    
    Args:
        data_dir: Path to HAR_Video_Dataset directory
        batch_size: Batch size for dataloaders
        max_frames: Maximum frames per video
        frame_size: Target frame size (H, W)
        num_workers: Number of worker processes
    
    Returns:
        train_loader, val_loader, test_loader, class_names
    """
    
    # Create datasets
    train_dataset = HARVideoDataset(
        data_dir=data_dir,
        split='train',
        max_frames=max_frames,
        frame_size=frame_size
    )
    
    val_dataset = HARVideoDataset(
        data_dir=data_dir,
        split='val', 
        max_frames=max_frames,
        frame_size=frame_size
    )
    
    test_dataset = HARVideoDataset(
        data_dir=data_dir,
        split='test',
        max_frames=max_frames,
        frame_size=frame_size
    )
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    return train_loader, val_loader, test_loader, train_dataset.classes

# Initialize the dataset
print("🎬 Initializing Human Action Recognition Dataset for WAN 2.1 training...")

data_dir = "/mnt/Storage02/SoftwareDev/CAP_6411_Assignments/Aug2025/01_Assignments/08_WAN_2_1/data/HAR_Video_Dataset"

# Check if dataset exists
if not os.path.exists(data_dir):
    print(f"❌ Dataset directory not found: {data_dir}")
else:
    print(f"✅ Found dataset directory: {data_dir}")
    
    # Create dataloaders with appropriate settings for WAN 2.1
    try:
        train_loader, val_loader, test_loader, class_names = create_har_dataloaders(
            data_dir=data_dir,
            batch_size=2,  # Small batch size for video data
            max_frames=16,  # 16 frames per video clip
            frame_size=(256, 256),  # Standard resolution for video models
            num_workers=2
        )
        
        print(f"\n📊 Dataset Statistics:")
        print(f"   Classes: {len(class_names)} ({', '.join(class_names)})")
        print(f"   Training samples: {len(train_loader.dataset)}")
        print(f"   Validation samples: {len(val_loader.dataset)}")
        print(f"   Test samples: {len(test_loader.dataset)}")
        print(f"   Batch size: {train_loader.batch_size}")
        print(f"   Max frames per video: 16")
        print(f"   Frame resolution: 256x256")
        
        # Test loading a batch
        print("\n🔄 Testing data loading...")
        sample_batch = next(iter(train_loader))
        
        print(f"   Batch shape - Frames: {sample_batch['frames'].shape}")
        print(f"   Batch labels: {sample_batch['label']}")
        print(f"   Sample prompts: {sample_batch['prompt'][:2]}")  # Show first 2 prompts
        
        print(f"\n✅ Dataset successfully prepared for WAN 2.1 training!")
        print(f"   - Video frames are normalized to [0,1] range")
        print(f"   - Frame format: (Batch, Channels, Time, Height, Width)")
        print(f"   - Text prompts generated for each action class") 
        print(f"   - Train/Val/Test splits created with stratification")
        
        # Save dataset info for reference
        dataset_info = {
            'classes': class_names,
            'num_classes': len(class_names),
            'train_size': len(train_loader.dataset),
            'val_size': len(val_loader.dataset), 
            'test_size': len(test_loader.dataset),
            'max_frames': 16,
            'frame_size': [256, 256],
            'batch_size': 2
        }

        os.chdir('..')
        filepath = 'data/dataset_info.json'

        # Create dataset_info.json file if it doesn't exist
        os.makedirs(os.path.dirname(filepath), exist_ok=True)

        with open(filepath, 'w') as f:
            json.dump(dataset_info, f, indent=2)

        print(f"\n💾 Dataset info saved to '{filepath}'")
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 
    except Exception as e:
        print(f"❌ Error creating dataloaders: {str(e)}")
        import traceback
        traceback.print_exc()

# Clear memory after dataset preparation
clear_memory()

🎬 Initializing Human Action Recognition Dataset for WAN 2.1 training...
✅ Found dataset directory: /mnt/Storage02/SoftwareDev/CAP_6411_Assignments/Aug2025/01_Assignments/08_WAN_2_1/data/HAR_Video_Dataset
Loaded 779 videos for train split
Classes: ['Clapping', 'Meet and Split', 'Sitting', 'Standing Still', 'Walking', 'Walking While Reading Book', 'Walking While Using Phone']
Class distribution: {'Clapping': 102, 'Meet and Split': 103, 'Sitting': 109, 'Standing Still': 122, 'Walking': 120, 'Walking While Reading Book': 123, 'Walking While Using Phone': 100}
Loaded 167 videos for val split
Classes: ['Clapping', 'Meet and Split', 'Sitting', 'Standing Still', 'Walking', 'Walking While Reading Book', 'Walking While Using Phone']
Class distribution: {'Clapping': 22, 'Meet and Split': 22, 'Sitting': 24, 'Standing Still': 26, 'Walking': 26, 'Walking While Reading Book': 26, 'Walking While Using Phone': 21}
Loaded 167 videos for test split
Classes: ['Clapping', 'Meet and Split', 'Sitting', 'Stan